# Yeu cau 2: Model Training & Tracking
Huan luyen Naive Bayes va XGBoost, track bang MLflow.

In [ ]:
# Cai dat thu vien (neu chay tren Colab)
# !pip install mlflow xgboost scikit-learn scipy -q

In [1]:
import mlflow
import mlflow.sklearn
import numpy as np
import scipy.sparse as sp
import pickle
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from xgboost import XGBClassifier

# Doc du lieu da xu ly
X = sp.load_npz('X_tfidf.npz')
y = np.load('y_labels.npy')

# Split du lieu: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}')

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 3618, Val: 775, Test: 776


In [2]:
def evaluate(model, X, y):
    pred = model.predict(X)
    return {
        'accuracy':  round(accuracy_score(y, pred), 4),
        'precision': round(precision_score(y, pred), 4),
        'recall':    round(recall_score(y, pred), 4),
        'f1':        round(f1_score(y, pred), 4)
    }

mlflow.set_experiment('spam_classification')

2026/05/04 10:38:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/04 10:38:34 INFO mlflow.store.db.utils: Updating database tables
2026/05/04 10:38:35 INFO mlflow.tracking.fluent: Experiment with name 'spam_classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///d:/StudyDocument/CongNgheMoi/22635801_NNL/mlruns/1', creation_time=1777865915327, experiment_id='1', last_update_time=1777865915327, lifecycle_stage='active', name='spam_classification', tags={}, workspace='default'>

In [3]:
# Model 1: Naive Bayes
with mlflow.start_run(run_name='naive_bayes'):
    params = {'alpha': 0.1}
    nb = MultinomialNB(alpha=params['alpha'])
    nb.fit(X_train, y_train)

    val_metrics  = evaluate(nb, X_val, y_val)
    test_metrics = evaluate(nb, X_test, y_test)

    mlflow.log_params(params)
    mlflow.log_metrics({f'val_{k}': v for k, v in val_metrics.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})
    mlflow.sklearn.log_model(nb, 'naive_bayes_model')

    print('Naive Bayes - Val:', val_metrics)
    print('Naive Bayes - Test:', test_metrics)

2026/05/04 10:38:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 10:38:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Naive Bayes - Val: {'accuracy': 0.9755, 'precision': 0.954, 'recall': 0.8469, 'f1': 0.8973}
Naive Bayes - Test: {'accuracy': 0.9794, 'precision': 0.9457, 'recall': 0.8878, 'f1': 0.9158}


In [4]:
# Model 2: XGBoost
with mlflow.start_run(run_name='xgboost'):
    params = {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
    xgb = XGBClassifier(**params, random_state=42, eval_metric='logloss', use_label_encoder=False)
    xgb.fit(X_train, y_train)

    val_metrics  = evaluate(xgb, X_val, y_val)
    test_metrics = evaluate(xgb, X_test, y_test)

    mlflow.log_params(params)
    mlflow.log_metrics({f'val_{k}': v for k, v in val_metrics.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})
    mlflow.sklearn.log_model(xgb, 'xgboost_model')

    print('XGBoost - Val:', val_metrics)
    print('XGBoost - Test:', test_metrics)

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [10:38:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/05/04 10:38:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 10:38:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost - Val: {'accuracy': 0.9381, 'precision': 1.0, 'recall': 0.5102, 'f1': 0.6757}
XGBoost - Test: {'accuracy': 0.9536, 'precision': 0.9697, 'recall': 0.6531, 'f1': 0.7805}


In [ ]:
# Xem MLflow UI (neu chay local)
# Chay lenh nay tren terminal: mlflow ui
# Sau do mo trinh duyet: http://127.0.0.1:5000
print('Chay: mlflow ui  ->  mo http://127.0.0.1:5000 de xem ket qua')